# Final Model: DistilBERT
**Nova IMS - Text Mining 2025/2026 - Group 31**

This notebook contains our final solution. It explicitly implements the single champion pipeline (DistilBERT), trains on the dataset, classifies the test dataset, and saves the predictions to `outputs/pred_31.csv` as requested by the rubric.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Setup Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

TEST_CSV_PATH = "data/test.csv"
PRED_CSV_PATH = "outputs/pred_31.csv"


### Module: `src/config.py`


In [ ]:
import re

# Reproducibility
SEED = 42

# Dataset splits
VAL_SIZE = 0.20
K_FOLD_N_SPLITS = 5

# Paths
TRAIN_CSV_PATH = "data/train.csv"
TEST_CSV_PATH = "data/test.csv"
RESULTS_CSV_PATH = "outputs/results.csv"
OUTPUT_PRED_PATH = "outputs/pred_best.csv"

# Error analysis output paths
CONF_MATRIX_PLOT_PATH = "outputs/confusion_matrix.png"
MISCLASSIFIED_TXT_PATH = "outputs/misclassified_report.txt"
MISCLASSIFIED_JSON_PATH = "outputs/misclassified_analysis.json"

# Labels
NUM_LABELS = 3
LABEL_NAMES = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
LABEL2ID = {"Bearish": 0, "Bullish": 1, "Neutral": 2}
ID2LABEL = {0: "Bearish", 1: "Bullish", 2: "Neutral"}

# DistilBERT
DISTILBERT_MODEL_NAME = "distilbert-base-uncased"
DISTILBERT_N_SAMPLES_SPIKE = 200
DISTILBERT_CACHE_DIR = "outputs/distilbert_cache"
DISTILBERT_CHECKPOINT_DIR = "outputs/distilbert_checkpoints"

# Qwen decoder
QWEN_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# Twitter-RoBERTa
ROBERTA_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
ROBERTA_N_SAMPLES_SPIKE = 200
ROBERTA_CACHE_DIR = "outputs/roberta_cache"
ROBERTA_CHECKPOINT_DIR = "outputs/roberta_checkpoints"

# FinBERT
FINBERT_MODEL_NAME = "ProsusAI/finbert"
FINBERT_N_SAMPLES_SPIKE = 200
FINBERT_CACHE_DIR = "outputs/finbert_cache"
FINBERT_CHECKPOINT_DIR = "outputs/finbert_checkpoints"

# DeBERTa-v3
DEBERTA_MODEL_NAME = "microsoft/deberta-v3-base"
DEBERTA_N_SAMPLES_SPIKE = 200
DEBERTA_CACHE_DIR = "outputs/deberta_cache"
DEBERTA_CHECKPOINT_DIR = "outputs/deberta_checkpoints"

# Feature matrix cache paths
BOW_TRAIN_PATH       = "outputs/X_train_bow.npz"
BOW_VAL_PATH         = "outputs/X_val_bow.npz"
TFIDF_UNI_TRAIN_PATH = "outputs/X_train_tfidf_uni.npz"
TFIDF_UNI_VAL_PATH   = "outputs/X_val_tfidf_uni.npz"
TFIDF_OPT_TRAIN_PATH = "outputs/X_train_tfidf_opt.npz"
TFIDF_OPT_VAL_PATH   = "outputs/X_val_tfidf_opt.npz"

# Leaderboard CSV schema
RESULTS_HEADERS = [
    "timestamp", "owner", "model_name", "feature_description",
    "accuracy", "precision_macro", "recall_macro", "f1_macro",
    "parameters", "f1_per_class", "notes",
]

# Preprocessing — placeholders
URL_PLACEHOLDER = "URL_PLACEHOLDER"
MENTION_PLACEHOLDER = "MENTION_PLACEHOLDER"
CASHTAG_PLACEHOLDER = "CASHTAG_PLACEHOLDER"
PROTECTED_PLACEHOLDERS = {URL_PLACEHOLDER, MENTION_PLACEHOLDER, CASHTAG_PLACEHOLDER}

# Preprocessing — financial noise tokens (used on top of NLTK stopwords)
FINANCIAL_STOPWORDS = {
    "rt", "amp", "co", "qt", "http", "https", "via",
    "stock", "stocks", "ticker", "tickers", "share", "shares",
}

# EDA — display
LABEL_PALETTE = {
    "Bearish": "#E06666",
    "Bullish": "#6AA84F",
    "Neutral": "#4A90E2",
}

# EDA — artifact detection patterns
URL_PATTERN     = re.compile(r'https?://\S+|www\.\S+')
MENTION_PATTERN = re.compile(r'@\w+')
CASHTAG_PATTERN = re.compile(r'\$\w+')
HASHTAG_PATTERN = re.compile(r'#\w+')

# EDA — fast stopword set (no NLTK dependency)
DEFAULT_STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
    "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which",
    "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be",
    "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an",
    "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by",
    "for", "with", "about", "against", "between", "into", "through", "during", "before",
    "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over",
    "under", "again", "further", "then", "once", "here", "there", "when", "where", "why",
    "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such",
    "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can",
    "will", "just", "don", "should", "now", "d", "ll", "m", "o", "re", "ve", "y",
    *FINANCIAL_STOPWORDS,
}


### Module: `src/utils.py`


In [ ]:
def log_info(msg: str) -> None:
    print(f"[INFO] {msg}")

def log_success(msg: str) -> None:
    print(f"[SUCCESS] {msg}")

def log_error(msg: str) -> None:
    print(f"[ERROR] {msg}")

def log_warning(msg: str) -> None:
    print(f"[WARNING] {msg}")

def print_header(title: str, width: int = 60) -> None:
    print("=" * width)
    print(title)
    print("=" * width)

def print_separator(width: int = 60) -> None:
    print("-" * width)


### Module: `src/evaluate.py`


In [ ]:
import os
import csv
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix,
)



def compute_metrics(y_true, y_pred) -> dict:
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    per_class = f1_score(y_true, y_pred, average=None, zero_division=0)
    f1_per_class = {
        name: float(per_class[i]) if i < len(per_class) else 0.0
        for i, name in LABEL_NAMES.items()
    }
    return {
        "accuracy": float(accuracy),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
        "f1_per_class": f1_per_class,
    }


def log_model_run(
    model_name: str,
    feature_desc: str,
    metrics: dict,
    params: str = "",
    owner: str = "",
    notes: str = "",
) -> None:
    """Logs a model run to results.csv idempotently — updates if the same key exists."""
    os.makedirs(os.path.dirname(RESULTS_CSV_PATH), exist_ok=True)

    new_row = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "owner": owner,
        "model_name": model_name,
        "feature_description": feature_desc,
        "accuracy": f"{metrics['accuracy']:.4f}",
        "precision_macro": f"{metrics['precision_macro']:.4f}",
        "recall_macro": f"{metrics['recall_macro']:.4f}",
        "f1_macro": f"{metrics['f1_macro']:.4f}",
        "parameters": params,
        "f1_per_class": str(metrics.get("f1_per_class", "")),
        "notes": notes,
    }

    def _is_match(row: dict) -> bool:
        return (
            row["model_name"] == model_name
            and row["feature_description"] == feature_desc
            and row["parameters"] == params
            and row["owner"] == owner
        )

    existing_rows = []
    updated = False

    if os.path.exists(RESULTS_CSV_PATH) and os.path.getsize(RESULTS_CSV_PATH) > 0:
        try:
            with open(RESULTS_CSV_PATH, mode="r", newline="", encoding="utf-8") as f:
                reader = csv.reader(f)
                next(reader, None)
                for r in reader:
                    if len(r) != len(RESULTS_HEADERS):
                        continue
                    row = dict(zip(RESULTS_HEADERS, r))
                    if _is_match(row):
                        existing_rows.append(new_row)
                        updated = True
                    else:
                        existing_rows.append(row)
        except Exception as e:
            log_warning(f"Error reading leaderboard CSV: {e}. Resetting.")
            existing_rows = []

    if not updated:
        existing_rows.append(new_row)

    with open(RESULTS_CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=RESULTS_HEADERS)
        writer.writeheader()
        writer.writerows(existing_rows)

    log_info(f"{'Updated' if updated else 'Added'} run in {RESULTS_CSV_PATH}")


def evaluate_and_log(
    y_true, y_pred,
    model_name: str,
    feature_desc: str,
    params: str = "",
    owner: str = "",
) -> dict:
    """Evaluates predictions, prints report, and logs to results.csv."""
    metrics = compute_metrics(y_true, y_pred)

    print_header(f"MODEL EVALUATION: {model_name} ({feature_desc})")
    log_info(f"Accuracy          : {metrics['accuracy']:.4f}")
    log_info(f"Precision (Macro) : {metrics['precision_macro']:.4f}")
    log_info(f"Recall (Macro)    : {metrics['recall_macro']:.4f}")
    log_info(f"F1 (Macro)        : {metrics['f1_macro']:.4f}")
    print_separator()
    print(classification_report(y_true, y_pred, target_names=list(LABEL_NAMES.values()), zero_division=0))
    print_separator()
    print(confusion_matrix(y_true, y_pred))

    log_model_run(model_name, feature_desc, metrics, params, owner=owner)
    return metrics


def evaluate_model(y_true, y_pred, owner: str, model, notes: str = "") -> dict:
    """Bento's interface — computes metrics, prints report, and logs to results.csv."""
    model_name = model.__class__.__name__ if hasattr(model, "__class__") else str(model)
    hyperparameters = str(model.get_params()) if hasattr(model, "get_params") else ""

    metrics = compute_metrics(y_true, y_pred)
    print(classification_report(y_true, y_pred, zero_division=0))
    print(confusion_matrix(y_true, y_pred))

    log_model_run(
        model_name=model_name,
        feature_desc="N/A",
        metrics=metrics,
        params=hyperparameters,
        owner=owner,
        notes=notes,
    )

    return {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "owner": owner,
        "model": model_name,
        "hyperparams": hyperparameters,
        "val_accuracy": metrics["accuracy"],
        "val_p_macro": metrics["precision_macro"],
        "val_r_macro": metrics["recall_macro"],
        "val_f1_macro": metrics["f1_macro"],
        "val_f1_per_class": metrics["f1_per_class"],
        "notes": notes,
    }


def save_submission(
    test_df: pd.DataFrame,
    predictions,
    output_path: str = OUTPUT_PRED_PATH,
    id_col: str = "id",
) -> pd.DataFrame:
    """Saves id + label predictions to CSV."""
    submission = pd.DataFrame({"id": test_df[id_col], "label": predictions})
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    submission.to_csv(output_path, index=False)
    log_success(f"Predictions saved to {output_path} ({len(submission)} rows)")
    return submission


### Module: `src/preprocessing.py`


In [ ]:
import re
import sys
import json
import argparse
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import TweetTokenizer
from nltk.stem import PorterStemmer, WordNetLemmatizer
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

    SEED, VAL_SIZE, K_FOLD_N_SPLITS,
    URL_PATTERN, MENTION_PATTERN, CASHTAG_PATTERN,
    URL_PLACEHOLDER, MENTION_PLACEHOLDER, CASHTAG_PLACEHOLDER, PROTECTED_PLACEHOLDERS,
    FINANCIAL_STOPWORDS,
)


def _download_nltk_resources() -> None:
    resources = {
        'stopwords': 'corpora/stopwords',
        'punkt': 'tokenizers/punkt',
        'wordnet': 'corpora/wordnet',
        'omw-1.4': 'corpora/omw-1.4',
    }
    for name, path in resources.items():
        try:
            nltk.data.find(path)
        except LookupError:
            nltk.download(name, quiet=True)


_download_nltk_resources()

_SPACE_RE = re.compile(r'\s+')
_UNICODE_MAP = [
    (re.compile(r'[\u201c\u201d\u201e\u201f]'), '"'),
    (re.compile(r'[\u2018\u2019\u201a\u201b]'), "'"),
    (re.compile(r'[\u2012\u2013\u2014\u2015\u2212]'), '-'),
    (re.compile(r'\u2026'), '...'),
    (re.compile(r'\uFFFD'), ' '),
]
_TOKENIZER = TweetTokenizer(preserve_case=True, reduce_len=True, strip_handles=False)

# ── Text preprocessing ────────────────────────────────────────────────────────

def normalize_unicode_punctuation(text: str) -> str:
    if not isinstance(text, str):
        return ""
    for pattern, replacement in _UNICODE_MAP:
        text = pattern.sub(replacement, text)
    return text


def clean_regex(
    text: str,
    url_mode: str = 'replace',
    mention_mode: str = 'replace',
    cashtag_mode: str = 'keep',
) -> str:
    if url_mode == 'remove':
        text = URL_PATTERN.sub('', text)
    elif url_mode == 'replace':
        text = URL_PATTERN.sub(URL_PLACEHOLDER, text)

    if mention_mode == 'remove':
        text = MENTION_PATTERN.sub('', text)
    elif mention_mode == 'replace':
        text = MENTION_PATTERN.sub(MENTION_PLACEHOLDER, text)

    if cashtag_mode == 'remove':
        text = CASHTAG_PATTERN.sub('', text)
    elif cashtag_mode == 'replace':
        text = CASHTAG_PATTERN.sub(CASHTAG_PLACEHOLDER, text)

    return _SPACE_RE.sub(' ', text).strip()


def tokenize_tweet(text: str) -> list:
    return _TOKENIZER.tokenize(text)


def remove_stopwords_from_tokens(tokens: list, extra_stopwords: list = None) -> list:
    all_stops = set(stopwords.words('english')) | FINANCIAL_STOPWORDS
    if extra_stopwords:
        all_stops.update(extra_stopwords)
    return [t for t in tokens if (t.lower() not in all_stops) or (t in PROTECTED_PLACEHOLDERS)]


def apply_stemming(tokens: list) -> list:
    stemmer = PorterStemmer()
    return [t if t in PROTECTED_PLACEHOLDERS else stemmer.stem(t) for t in tokens]


def apply_lemmatization(tokens: list) -> list:
    lemmatizer = WordNetLemmatizer()
    return [t if t in PROTECTED_PLACEHOLDERS else lemmatizer.lemmatize(t) for t in tokens]


def preprocess_tweet(
    text: str,
    lowercase: bool = True,
    url_mode: str = 'replace',
    mention_mode: str = 'replace',
    cashtag_mode: str = 'keep',
    remove_stopwords: bool = True,
    custom_stopwords: list = None,
    use_stemming: bool = False,
    use_lemmatization: bool = True,
    return_str: bool = False,
):
    """Full tweet preprocessing pipeline. Returns token list or joined string."""
    text = normalize_unicode_punctuation(text)
    if lowercase:
        text = text.lower()
    text = clean_regex(text, url_mode=url_mode, mention_mode=mention_mode, cashtag_mode=cashtag_mode)
    tokens = tokenize_tweet(text)
    if remove_stopwords:
        tokens = remove_stopwords_from_tokens(tokens, extra_stopwords=custom_stopwords)
    if use_stemming:
        tokens = apply_stemming(tokens)
    if use_lemmatization:
        tokens = apply_lemmatization(tokens)
    return " ".join(tokens) if return_str else tokens


# ── Train / val splitting (absorbed from train_val_split.py) ──────────────────

def stratified_split(
    dataset: pd.DataFrame,
    test_size: float = VAL_SIZE,
    seed: int = SEED,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
    X, y = dataset['text'], dataset['label']
    return train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)


def create_stratified_kfold(n_splits: int = K_FOLD_N_SPLITS, seed: int = SEED) -> StratifiedKFold:
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)


# ── Smoke tests ───────────────────────────────────────────────────────────────

def run_smoke_tests() -> None:
    """Verifies each preprocessing step with representative tweets."""
    print_header("PREPROCESSING SMOKE TESTS")

    test_tweets = [
        "Downgrades 4/7: $MLND to underperform at Needham—see details... https://t.co/example",
        "Shorting $AAPL here at $180. @elonmusk thoughts? #market #trading",
        "RT @NovaIMS: $BTC is falling down rapidly... RT to warn others!",
        "Having a cup of coffee and watching the market open. Very neutral.",
    ]

    for idx, raw in enumerate(test_tweets, 1):
        log_info(f"Test {idx}: '{raw}'")
        lem = preprocess_tweet(raw, return_str=True)
        log_info(f"  Lemmatized : '{lem}'")
        stem = preprocess_tweet(raw, url_mode='remove', mention_mode='remove',
                                cashtag_mode='replace', use_stemming=True,
                                use_lemmatization=False, return_str=False)
        log_info(f"  Stemmed    : {stem}")

    log_success("Smoke tests complete.")


# ── Main Entry Point for CLI & Agents ─────────────────────────────────────────

if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description="Text Mining Preprocessing Pipeline Tool. Highly configurable for direct command-line use or integration as an agent tool."
    )
    
    # Text input options
    input_group = parser.add_mutually_exclusive_group()
    input_group.add_argument('--text', type=str, help="Single tweet text to preprocess.")
    input_group.add_argument('--smoke-tests', action='store_true', help="Run verification smoke tests.")
    input_group.add_argument('--json-input', action='store_true', help="Enable JSON input via stdin (great for agent workflows).")

    # Pipeline configurations
    parser.add_argument('--no-lowercase', action='store_false', dest='lowercase', help="Disable lowercasing.")
    parser.add_argument('--url-mode', type=str, choices=['keep', 'remove', 'replace'], default='replace',
                        help="How to handle URLs. Default: 'replace' with 'URL_PLACEHOLDER'.")
    parser.add_argument('--mention-mode', type=str, choices=['keep', 'remove', 'replace'], default='replace',
                        help="How to handle mentions (@user). Default: 'replace' with 'MENTION_PLACEHOLDER'.")
    parser.add_argument('--cashtag-mode', type=str, choices=['keep', 'remove', 'replace'], default='keep',
                        help="How to handle cashtags ($TSLA). Default: 'keep'.")
    parser.add_argument('--no-stopwords', action='store_false', dest='remove_stopwords', help="Disable stopwords removal.")
    parser.add_argument('--custom-stopwords', type=str, nargs='*', help="List of extra custom stopwords to remove.")
    parser.add_argument('--stem', action='store_true', dest='use_stemming', help="Apply Porter stemming.")
    parser.add_argument('--no-lemmatization', action='store_false', dest='use_lemmatization', help="Disable WordNet lemmatization.")
    parser.add_argument('--return-str', action='store_true', help="Return results as a space-joined string instead of a token list.")

    args = parser.parse_args()

    # Case 1: Run Smoke Tests
    if args.smoke_tests or len(sys.argv) == 1:
        run_smoke_tests()
        sys.exit(0)

    # Case 2: Read JSON from stdin for Agent integration
    if args.json_input:
        try:
            input_data = json.load(sys.stdin)
            text_to_process = input_data.get('text', '')
            
            # Allow JSON keys to override arguments
            lowercase = input_data.get('lowercase', args.lowercase)
            url_mode = input_data.get('url_mode', args.url_mode)
            mention_mode = input_data.get('mention_mode', args.mention_mode)
            cashtag_mode = input_data.get('cashtag_mode', args.cashtag_mode)
            remove_stopwords = input_data.get('remove_stopwords', args.remove_stopwords)
            custom_stopwords = input_data.get('custom_stopwords', args.custom_stopwords)
            use_stemming = input_data.get('use_stemming', args.use_stemming)
            use_lemmatization = input_data.get('use_lemmatization', args.use_lemmatization)
            return_str = input_data.get('return_str', args.return_str)
            
            result = preprocess_tweet(
                text=text_to_process,
                lowercase=lowercase,
                url_mode=url_mode,
                mention_mode=mention_mode,
                cashtag_mode=cashtag_mode,
                remove_stopwords=remove_stopwords,
                custom_stopwords=custom_stopwords,
                use_stemming=use_stemming,
                use_lemmatization=use_lemmatization,
                return_str=return_str
            )
            
            print(json.dumps({"status": "success", "result": result}))
        except Exception as e:
            print(json.dumps({"status": "error", "message": str(e)}))
        sys.exit(0)

    # Case 3: Process single string passed via --text
    if args.text:
        result = preprocess_tweet(
            text=args.text,
            lowercase=args.lowercase,
            url_mode=args.url_mode,
            mention_mode=args.mention_mode,
            cashtag_mode=args.cashtag_mode,
            remove_stopwords=args.remove_stopwords,
            custom_stopwords=args.custom_stopwords,
            use_stemming=args.use_stemming,
            use_lemmatization=args.use_lemmatization,
            return_str=args.return_str
        )
        print(result)


### Module: `src/distilbert_trainer.py`


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

    SEED, TRAIN_CSV_PATH as DATA_PATH, TEST_CSV_PATH, NUM_LABELS, LABEL2ID, ID2LABEL,
    DISTILBERT_MODEL_NAME as MODEL_NAME,
    DISTILBERT_CACHE_DIR, DISTILBERT_CHECKPOINT_DIR,
    DISTILBERT_N_SAMPLES_SPIKE as N_SAMPLES_SPIKE,
)


# ── Environment helpers ──────────────────────────────────────────────────────

def check_gpu() -> str:
    if torch.cuda.is_available():
        log_info(f"CUDA available — {torch.cuda.get_device_name(0)}")
        return "cuda"
    log_info("No CUDA device found — using CPU")
    return "cpu"


def load_tokenizer() -> DistilBertTokenizerFast:
    log_info(f"Loading tokenizer: {MODEL_NAME}")
    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
    log_info(f"Vocab size: {tokenizer.vocab_size}")
    return tokenizer


def tokenize_samples(tokenizer: DistilBertTokenizerFast, texts: list[str]) -> dict:
    return tokenizer(texts, padding=True, truncation=True, max_length=128, return_tensors="pt")


def run_spike() -> None:
    device = check_gpu()
    tokenizer = load_tokenizer()

    sample_texts = pd.read_csv(DATA_PATH)["text"].head(N_SAMPLES_SPIKE).tolist()
    log_info(f"Loaded {len(sample_texts)} samples from {DATA_PATH}")

    encoded = {k: v.to(device) for k, v in tokenize_samples(tokenizer, sample_texts).items()}

    log_info(f"input_ids shape        : {tuple(encoded['input_ids'].shape)}")
    log_info(f"attention_mask shape   : {tuple(encoded['attention_mask'].shape)}")
    log_info(f"Sample tokens (row 0)  : {encoded['input_ids'][0, :10].tolist()} ...")
    log_success("Spike complete — environment is ready for DistilBERT fine-tuning.")


# ── Training pipeline ─────────────────────────────────────────────────────────

def load_data(n_samples: int | None = None) -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    if n_samples:
        df = df.sample(n=n_samples, random_state=SEED).reset_index(drop=True)
    return df


def build_hf_datasets(
    tokenizer: DistilBertTokenizerFast,
    X_train: pd.Series,
    X_val: pd.Series,
    y_train: pd.Series,
    y_val: pd.Series,
    cache_dir: Path,
) -> tuple[Dataset, Dataset]:
    cache_dir.mkdir(parents=True, exist_ok=True)

    def _tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    def _build(texts, labels, cache_path):
        if cache_path.exists():
            log_info(f"Loading dataset from cache: {cache_path}")
            return Dataset.load_from_disk(str(cache_path))
        ds = Dataset.from_dict({"text": texts.tolist(), "label": labels.tolist()})
        ds = ds.map(_tokenize, batched=True, remove_columns=["text"])
        ds.save_to_disk(str(cache_path))
        log_info(f"Dataset cached to: {cache_path}")
        return ds

    train_ds = _build(X_train, y_train, cache_dir / "train")
    val_ds   = _build(X_val,   y_val,   cache_dir / "val")
    return train_ds, val_ds


def make_compute_metrics(owner: str = "", notes: str = ""):
    def _compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        metrics = compute_metrics(labels, preds)
        log_model_run(
            model_name="DistilBERT",
            feature_desc="HF fine-tune",
            metrics=metrics,
            params=f"model={MODEL_NAME}, max_length=128",
            owner=owner,
            notes=notes,
        )
        return {"accuracy": metrics["accuracy"], "f1_macro": metrics["f1_macro"]}
    return _compute_metrics


def build_trainer(
    model: DistilBertForSequenceClassification,
    tokenizer: DistilBertTokenizerFast,
    train_ds: Dataset,
    val_ds: Dataset,
    output_dir: str | None = None,
    learning_rate: float = 2e-5,
    per_device_batch_size: int = 16,
    num_epochs: int = 3,
    owner: str = "",
    notes: str = "",
) -> Trainer:
    training_args = TrainingArguments(
        output_dir=output_dir or str(Path(DISTILBERT_CHECKPOINT_DIR) / MODEL_NAME.replace("/", "_")),
        num_train_epochs=num_epochs,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        learning_rate=learning_rate,
        weight_decay=0.01,
        warmup_ratio=0.1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=SEED,
        logging_steps=50,
        report_to="none",
    )
    return Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=make_compute_metrics(owner=owner, notes=notes),
    )


def run_trainer(n_samples: int | None = 500, owner: str = "", notes: str = "") -> Trainer:
    log_info(f"Loading {'all' if n_samples is None else n_samples} samples ...")
    df = load_data(n_samples=n_samples)
    X_train, X_val, y_train, y_val = stratified_split(df)
    log_info(f"Split — train={len(X_train)}, val={len(X_val)}")

    tokenizer = load_tokenizer()

    # Use a sample-size-specific cache dir so a 500-sample spike doesn't get
    # served when we ask for the full dataset.
    cache_suffix = "full" if n_samples is None else f"n{n_samples}"
    cache_dir = Path(DISTILBERT_CACHE_DIR) / cache_suffix
    train_ds, val_ds = build_hf_datasets(tokenizer, X_train, X_val, y_train, y_val, cache_dir)
    log_info(f"Datasets — train={len(train_ds)} rows, val={len(val_ds)} rows")

    log_info("Loading DistilBertForSequenceClassification ...")
    model = DistilBertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, id2label=ID2LABEL, label2id=LABEL2ID,
    )

    trainer = build_trainer(model=model, tokenizer=tokenizer,
                            train_ds=train_ds, val_ds=val_ds,
                            owner=owner, notes=notes)

    log_info("Starting training ...")
    trainer.train()
    log_info("Running final evaluation ...")
    trainer.evaluate()
    log_success("Training complete.")
    return trainer


def predict_test_set(
    trainer: Trainer,
    tokenizer: DistilBertTokenizerFast,
    test_csv_path: str = TEST_CSV_PATH,
) -> np.ndarray:
    log_info(f"Loading test set from {test_csv_path}")
    test_df = pd.read_csv(test_csv_path)
    log_info(f"Test rows: {len(test_df)}")

    def _tokenize(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    test_ds = Dataset.from_dict({"text": test_df["text"].tolist()})
    test_ds = test_ds.map(_tokenize, batched=True, remove_columns=["text"])

    preds = trainer.predict(test_ds)
    labels = np.argmax(preds.predictions, axis=-1)
    log_success(f"Generated {len(labels)} test predictions.")
    return labels


### Execute Final Prediction Logic

In [ ]:
log_info("Starting the official DistilBERT champion pipeline...")
trainer = run_trainer(n_samples=None, owner="Final Pipeline")
tokenizer = load_tokenizer()

log_info("Running predictions on the test set...")
preds = predict_test_set(trainer, tokenizer, test_csv_path=TEST_CSV_PATH)

test_df = pd.read_csv(TEST_CSV_PATH)
save_submission(test_df, preds, output_path=PRED_CSV_PATH, id_col="id")
print("\nPredictions successfully saved to:", PRED_CSV_PATH)
